In [ ]:
"""
VIF (Variance Inflation Factor) Multicollinearity Diagnostic
================================================================
Dissertation: Predicting Revenue Growth and Cost Reduction from Business AI Adoption
 
PURPOSE
-------
Quantifies multicollinearity among the retained numeric predictor variables,
specifically to investigate the correlation cluster identified during EDA
around ai_maturity_score (correlated with ai_adoption_rate, ai_training_hours,
ai_budget_percentage, ai_projects_active, and productivity_change_percent).
 
VIF measures how much a predictor's variance is inflated due to correlation
with other predictors. Common thresholds:
    VIF < 5   — low concern
    VIF 5-10  — moderate concern, worth noting
    VIF > 10  — high concern, often considered problematic
 
This is a DIAGNOSTIC step, not a modelling change — it quantifies the
multicollinearity already identified via the correlation matrix, giving a
citable, standard statistic for the Discussion chapter's limitations section
rather than a bare correlation coefficient alone.
 
Run: python vif_check.py
"""
 
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
 
DATA_PATH = "ai_company_adoption.csv"  # adjust to your actual file location
 
NUMERIC_FEATURES = [
    "ai_adoption_rate", "ai_maturity_score", "ai_budget_percentage",
    "ai_investment_per_employee", "ai_training_hours", "years_using_ai",
    "ai_projects_active", "task_automation_rate", "productivity_change_percent",
    "ai_risk_management_score", "regulatory_compliance_score",
    "annual_revenue_usd_millions", "num_employees",
]
# time_saved_per_week already excluded per Section 3.5 — not included here
 
df = pd.read_csv(DATA_PATH)
X = df[NUMERIC_FEATURES].dropna()
X_with_const = add_constant(X)
 
vif_data = pd.DataFrame()
vif_data["feature"] = X_with_const.columns
vif_data["VIF"] = [
    variance_inflation_factor(X_with_const.values, i)
    for i in range(X_with_const.shape[1])
]
vif_data = vif_data[vif_data["feature"] != "const"].sort_values("VIF", ascending=False)
 
print("=" * 70)
print("VARIANCE INFLATION FACTOR (VIF) — retained numeric predictors")
print("=" * 70)
print(vif_data.to_string(index=False))
 
print("\n--- Interpretation guide ---")
print("VIF < 5   : low concern")
print("VIF 5-10  : moderate concern")
print("VIF > 10  : high concern")
 
high_vif = vif_data[vif_data["VIF"] > 5]
if len(high_vif) > 0:
    print(f"\n⚠ {len(high_vif)} feature(s) show VIF > 5:")
    print(high_vif.to_string(index=False))
else:
    print("\n✅ No features exceed VIF = 5.")
 
vif_data.to_csv("vif_results.csv", index=False)
print("\nSaved: vif_results.csv")
 
print("\n--- Suggested dissertation text (Discussion, Limitations) ---")
top_vif = vif_data.iloc[0]
print(f"""
'A Variance Inflation Factor (VIF) analysis of the retained numeric predictors
confirmed the multicollinearity identified during exploratory data analysis:
{top_vif['feature']} showed the highest VIF ({top_vif['VIF']:.2f}), consistent
with its correlation with several related AI adoption variables (Section 4.1).
While tree-based models are comparatively robust to multicollinearity for
prediction purposes, SHAP-based feature attribution can be less stable when
predictors are highly correlated, meaning the relative importance attributed
to individual correlated features — particularly ai_maturity_score and
productivity_change_percent — should be interpreted with some caution.'
""")

In [ ]:
# VIF recomputed after excluding ai_maturity_score (Section 3.4)

FINAL_NUMERIC = [f for f in NUMERIC_FEATURES if f != "ai_maturity_score"]

X_after = add_constant(df[FINAL_NUMERIC].dropna())
vif_after = pd.DataFrame({
    "feature": X_after.columns,
    "VIF_after": [variance_inflation_factor(X_after.values, i)
                  for i in range(X_after.shape[1])]
}).query("feature != 'const'")

# Merge with the before values for a side-by-side table
comparison = (vif_data.query("feature != 'const'")
              .rename(columns={"VIF": "VIF_before"})
              .merge(vif_after, on="feature", how="outer")
              .sort_values("VIF_before", ascending=False))

print("=" * 70)
print("VIF BEFORE AND AFTER EXCLUDING ai_maturity_score")
print("=" * 70)
print(comparison.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))

n_high = (comparison["VIF_after"] > 10).sum()
print(f"\nPredictors with VIF > 10 after exclusion: {n_high}")

comparison.to_csv("vif_before_after.csv", index=False)
print("Saved: vif_before_after.csv")